In [ ]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm

# === CONFIG ===
image_root = "/home/ajarrah/PhD_Thesis/chapter_4/images/lipids/"
output_anndata_path = "/home/ajarrah/PhD_Thesis/chapter_4/msi_h5ad/aggregated_heatmap_grid_20um.h5ad"
square_size_um = 20.0   # µm
pixel_size_um = 1.0     # µm per pixel (adjust if needed)
sample_name = "aad_1"    # example sample folder to read

# === Define custom colormap ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),
    (0.00000001, "#000000"),
    (0.10, "#000080"),
    (0.15, "#0000FF"),
    (0.30, "#8000FF"),
    (0.45, "#FF0000"),
    (0.60, "#FF8000"),
    (0.75, "#FFFF00"),
    (1.0, "#FFFFFF")
])

# === Prepare LUT for color-to-value mapping ===
def build_cmap_LUT(cmap, n=2048):
    vals = np.linspace(0, 1, n)
    rgb = (cmap(vals)[:, :3] * 255).astype(np.uint8)
    return rgb, vals

cmap_LUT, cmap_vals = build_cmap_LUT(custom_cmap)

def rgb_to_value(rgb, cmap_LUT, cmap_vals):
    """Convert RGB pixels to normalized [0–1] scale based on nearest color in LUT."""
    rgb = rgb.reshape(-1, 3)
    # Compute Euclidean distance between pixel color and LUT colors
    diffs = np.sum((cmap_LUT[None, :, :] - rgb[:, None, :]) ** 2, axis=2)
    nearest = np.argmin(diffs, axis=1)
    return cmap_vals[nearest]

# === Get all PNG files for one sample ===
sample_dir = os.path.join(image_root, sample_name)
png_files = [f for f in os.listdir(sample_dir) if f.endswith(".png")]

# === Parse filename pattern: gene|sample|vmin|vmax.png ===
def parse_metadata(filename):
    match = re.match(r"(.+?)\|(.+?)\|([-\d\.]+)\|([-\d\.]+)\.png", filename)
    if match:
        gene, sample, vmin, vmax = match.groups()
        return gene, float(vmin), float(vmax)
    return None

# === Prepare grid coordinates ===
def make_grid(img_width, img_height, pixel_size_um, square_size_um):
    step = square_size_um / pixel_size_um
    xs = np.arange(0, img_width, step)
    ys = np.arange(0, img_height, step)
    xv, yv = np.meshgrid(xs, ys)
    return xv, yv

# === Main image processing ===
gene_list = []
expr_list = []
coords_list = None

for file in tqdm(png_files, desc=f"Processing {sample_name}"):
    meta = parse_metadata(file)
    if meta is None:
        continue
    gene, vmin, vmax = meta

    # Load heatmap image
    img_path = os.path.join(sample_dir, file)
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img)

    # Convert colors to expression values
    norm_vals = rgb_to_value(arr, cmap_LUT, cmap_vals)
    expr_img = norm_vals.reshape(arr.shape[:2]) * (vmax - vmin) + vmin

    # Create probing grid
    xv, yv = make_grid(arr.shape[1], arr.shape[0], pixel_size_um, square_size_um)
    half_size_px = int((square_size_um / pixel_size_um) / 2)
    grid_values = []

    for gx, gy in zip(xv.ravel(), yv.ravel()):
        x0 = int(max(gx - half_size_px, 0))
        x1 = int(min(gx + half_size_px, arr.shape[1]))
        y0 = int(max(gy - half_size_px, 0))
        y1 = int(min(gy + half_size_px, arr.shape[0]))
        region = expr_img[y0:y1, x0:x1]
        if region.size == 0:
            grid_values.append(np.nan)
        else:
            grid_values.append(np.nanmean(region))

    grid_values = np.array(grid_values)
    expr_list.append(grid_values)
    gene_list.append(gene)

    # Save grid coordinates once
    if coords_list is None:
        coords_list = np.column_stack((xv.ravel(), yv.ravel()))

# === Build AnnData ===
X = np.column_stack(expr_list)
obs = pd.DataFrame(coords_list, columns=["x_um", "y_um"])
var = pd.DataFrame(index=gene_list)

adata_grid = sc.AnnData(X=X, obs=obs, var=var)
adata_grid.X = np.nan_to_num(adata_grid.X, nan=0.0)

# === Save ===
adata_grid.write_h5ad(output_anndata_path)
print(f"✅ Grid-based AnnData saved at: {output_anndata_path}")
print(f"Shape: {adata_grid.shape[0]} squares × {adata_grid.shape[1]} genes")


Processing aad_1:   0%|          | 0/528 [00:00<?, ?it/s]

Processing aad_1:   1%|          | 3/528 [05:12<14:54:15, 102.20s/it]

In [6]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm
import multiprocessing as mp
from functools import partial
import resource

# === CONFIG ===
image_root = "/home/ajarrah/PhD_Thesis/chapter_4/images/lipids/"
output_anndata_path = "/home/ajarrah/PhD_Thesis/chapter_4/msi_h5ad/aggregated_heatmap_grid_20um.h5ad"
square_size_um = 20.0  # µm
pixel_size_um = 1.0  # µm per pixel (adjust if needed)
sample_name = "aad_1"  # example sample folder to read

# === RESOURCE LIMITS ===
MAX_WORKERS = 3 # Number of CPU cores to use
MAX_MEMORY_GB = 50  # Maximum memory per process in GB

# === Define custom colormap ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),
    (0.00000001, "#000000"),
    (0.10, "#000080"),
    (0.15, "#0000FF"),
    (0.30, "#8000FF"),
    (0.45, "#FF0000"),
    (0.60, "#FF8000"),
    (0.75, "#FFFF00"),
    (1.0, "#FFFFFF")
])

# === Prepare LUT for color-to-value mapping ===
def build_cmap_LUT(cmap, n=2048):
    vals = np.linspace(0, 1, n)
    rgb = (cmap(vals)[:, :3] * 255).astype(np.uint8)
    return rgb, vals

def rgb_to_value(rgb, cmap_LUT, cmap_vals):
    """Convert RGB pixels to normalized [0–1] scale based on nearest color in LUT."""
    rgb = rgb.reshape(-1, 3)
    # Compute Euclidean distance between pixel color and LUT colors
    diffs = np.sum((cmap_LUT[None, :, :] - rgb[:, None, :]) ** 2, axis=2)
    nearest = np.argmin(diffs, axis=1)
    return cmap_vals[nearest]

def parse_metadata(filename):
    """Parse filename pattern: gene|sample|vmin|vmax.png"""
    match = re.match(r"(.+?)\|(.+?)\|([-\d\.]+)\|([-\d\.]+)\.png", filename)
    if match:
        gene, sample, vmin, vmax = match.groups()
        return gene, float(vmin), float(vmax)
    return None

def make_grid(img_width, img_height, pixel_size_um, square_size_um):
    """Create grid coordinates for sampling."""
    step = square_size_um / pixel_size_um
    xs = np.arange(0, img_width, step)
    ys = np.arange(0, img_height, step)
    xv, yv = np.meshgrid(xs, ys)
    return xv, yv

def set_memory_limit(max_memory_gb):
    """Set memory limit for the current process."""
    try:
        max_memory_bytes = max_memory_gb * 1024 * 1024 * 1024
        resource.setrlimit(resource.RLIMIT_AS, (max_memory_bytes, max_memory_bytes))
    except (ValueError, OSError) as e:
        print(f"Warning: Could not set memory limit: {e}")

def process_single_image(file, sample_dir, cmap_LUT, cmap_vals, 
                         pixel_size_um, square_size_um, max_memory_gb):
    """Process a single PNG file and return gene name, expression values, and coordinates."""
    # Set memory limit for this process
    set_memory_limit(max_memory_gb)
    
    try:
        meta = parse_metadata(file)
        if meta is None:
            return None
        
        gene, vmin, vmax = meta
        
        # Load heatmap image
        img_path = os.path.join(sample_dir, file)
        img = Image.open(img_path).convert("RGB")
        arr = np.array(img)
        
        # Convert colors to expression values
        norm_vals = rgb_to_value(arr, cmap_LUT, cmap_vals)
        expr_img = norm_vals.reshape(arr.shape[:2]) * (vmax - vmin) + vmin
        
        # Create probing grid
        xv, yv = make_grid(arr.shape[1], arr.shape[0], pixel_size_um, square_size_um)
        half_size_px = int((square_size_um / pixel_size_um) / 2)
        
        grid_values = []
        coords = []
        
        for gx, gy in zip(xv.ravel(), yv.ravel()):
            x0 = int(max(gx - half_size_px, 0))
            x1 = int(min(gx + half_size_px, arr.shape[1]))
            y0 = int(max(gy - half_size_px, 0))
            y1 = int(min(gy + half_size_px, arr.shape[0]))
            
            region = expr_img[y0:y1, x0:x1]
            if region.size == 0:
                grid_values.append(np.nan)
            else:
                grid_values.append(np.nanmean(region))
            
            coords.append((gx, gy))
        
        grid_values = np.array(grid_values)
        coords = np.array(coords)
        
        return {
            'gene': gene,
            'values': grid_values,
            'coords': coords
        }
    
    except Exception as e:
        print(f"Error processing {file}: {e}")
        return None

def main():
    print(f"Starting multiprocessing with {MAX_WORKERS} workers and {MAX_MEMORY_GB}GB memory limit per process")
    
    # Get all PNG files for one sample
    sample_dir = os.path.join(image_root, sample_name)
    png_files = [f for f in os.listdir(sample_dir) if f.endswith(".png")]
    print(f"Found {len(png_files)} PNG files to process")
    
    # Build colormap LUT (shared across processes)
    cmap_LUT, cmap_vals = build_cmap_LUT(custom_cmap)
    
    # Create partial function with fixed parameters
    process_func = partial(
        process_single_image,
        sample_dir=sample_dir,
        cmap_LUT=cmap_LUT,
        cmap_vals=cmap_vals,
        pixel_size_um=pixel_size_um,
        square_size_um=square_size_um,
        max_memory_gb=MAX_MEMORY_GB
    )
    
    # Process images in parallel
    gene_list = []
    expr_list = []
    coords_list = None
    
    with mp.Pool(processes=MAX_WORKERS) as pool:
        results = list(tqdm(
            pool.imap(process_func, png_files),
            total=len(png_files),
            desc=f"Processing {sample_name}"
        ))
    
    # Collect results
    for result in results:
        if result is not None:
            gene_list.append(result['gene'])
            expr_list.append(result['values'])
            
            # Save grid coordinates once
            if coords_list is None:
                coords_list = result['coords']
    
    if not gene_list:
        print("No valid images processed!")
        return
    
    # === Build AnnData ===
    print("Building AnnData object...")
    X = np.column_stack(expr_list)
    obs = pd.DataFrame(coords_list, columns=["x_um", "y_um"])
    var = pd.DataFrame(index=gene_list)
    
    adata_grid = sc.AnnData(X=X, obs=obs, var=var)
    adata_grid.X = np.nan_to_num(adata_grid.X, nan=0.0)
    
    # === Save ===
    print("Saving AnnData...")
    adata_grid.write_h5ad(output_anndata_path)
    print(f"✅ Grid-based AnnData saved at: {output_anndata_path}")
    print(f"Shape: {adata_grid.shape[0]} squares × {adata_grid.shape[1]} genes")

if __name__ == "__main__":
    main()

Starting multiprocessing with 3 workers and 50GB memory limit per process
Found 528 PNG files to process


Processing aad_1: 100%|██████████| 528/528 [4:54:28<00:00, 33.46s/it]  


Building AnnData object...


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving AnnData...


FileNotFoundError: [Errno 2] Unable to create file (unable to open file: name = '/home/ajarrah/PhD_Thesis/chapter_4/msi_h5ad/aggregated_heatmap_grid_20um.h5ad', errno = 2, error message = 'No such file or directory', flags = 13, o_flags = 242)

In [7]:
adata_grid.write_h5ad(output_anndata_path)

NameError: name 'adata_grid' is not defined